<a href="https://colab.research.google.com/github/Priyadharshini-7431/SentimentScope/blob/main/SentimentScope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
import numpy as np

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q transformers datasets scikit-learn matplotlib

In [ ]:
import transformers
import datasets
import sklearn

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Setup completed successfully!")

Transformers: 5.18.0
Datasets: 4.8.5
Setup completed successfully!


In [ ]:


from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)
print("Training samples:", len(dataset["train"]))
print("Test samples:", len(dataset["test"]))

print("\nSample review:")
print(dataset["train"][0]["text"][:500])

print("\nSentiment label:", dataset["train"][0]["label"])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Training samples: 25000
Test samples: 25000

Sample review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent

Sentiment label: 0


In [ ]:

from sklearn.model_selection import train_test_split

# Original IMDb training data
train_data = dataset["train"]

# Split into training and validation (80% / 20%)
split_data = train_data.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split_data["train"]
val_dataset = split_data["test"]
test_dataset = dataset["test"]

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Training samples: 20000
Validation samples: 5000
Test samples: 25000


In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

sample_text = train_dataset[0]["text"]
tokens = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=128
)

print("Input IDs length:", len(tokens["input_ids"]))
print("Attention mask length:", len(tokens["attention_mask"]))
print("Sample label:", train_dataset[0]["label"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Input IDs length: 128
Attention mask length: 128
Sample label: 1


In [ ]:

import torch
from torch.utils.data import Dataset

class IMDbDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]

        encoding = self.tokenizer(
            item["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_torch_dataset = IMDbDataset(train_dataset, tokenizer)
val_torch_dataset = IMDbDataset(val_dataset, tokenizer)
test_torch_dataset = IMDbDataset(test_dataset, tokenizer)

print("PyTorch datasets created successfully!")
print("Training size:", len(train_torch_dataset))
print("Validation size:", len(val_torch_dataset))
print("Test size:", len(test_torch_dataset))

sample = train_torch_dataset[0]
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["labels"].item())

PyTorch datasets created successfully!
Training size: 20000
Validation size: 5000
Test size: 25000
Input IDs shape: torch.Size([128])
Attention mask shape: torch.Size([128])
Label: 1


In [ ]:

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Check one batch
batch = next(iter(train_loader))

print("\nInput IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

DataLoaders created successfully!
Training batches: 1250
Validation batches: 313
Test batches: 1563

Input IDs shape: torch.Size([16, 128])
Attention mask shape: torch.Size([16, 128])
Labels shape: torch.Size([16])


In [ ]:

from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = model.to(device)

print("Model loaded successfully!")
print("Device:", device)
print("Number of classes:", model.config.num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Device: cuda
Number of classes: 2


In [ ]:

EPOCHS = 2
LEARNING_RATE = 2e-5

In [ ]:

from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = CrossEntropyLoss()

print("Optimizer created successfully!")
print("Loss function created successfully!")

Optimizer created successfully!
Loss function created successfully!


In [ ]:

from tqdm.auto import tqdm
import torch

EPOCHS = 2

def calculate_accuracy(predictions, labels):
    predicted_labels = torch.argmax(predictions, dim=1)
    correct = (predicted_labels == labels).sum().item()
    return correct / labels.size(0)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = torch.argmax(outputs.logits, dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = total_correct / total_samples

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_samples = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)
            val_loss += loss.item()

            predictions = torch.argmax(outputs.logits, dim=1)
            val_correct += (predictions == labels).sum().item()
            val_samples += labels.size(0)

    val_loss /= len(val_loader)
    val_accuracy = val_correct / val_samples

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

Epoch 1/2:   0%|          | 0/1250 [00:00<?, ?it/s]

Validating:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 1/2
Training Loss: 0.3601
Training Accuracy: 83.90%
Validation Loss: 0.3160
Validation Accuracy: 85.94%


Epoch 2/2:   0%|          | 0/1250 [00:00<?, ?it/s]

Validating:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 2/2
Training Loss: 0.2206
Training Accuracy: 90.84%
Validation Loss: 0.3374
Validation Accuracy: 86.58%


In [ ]:

model.eval()

test_loss = 0.0
test_correct = 0
test_samples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        test_loss += loss.item()

        predictions = torch.argmax(outputs.logits, dim=1)
        test_correct += (predictions == labels).sum().item()
        test_samples += labels.size(0)

test_loss = test_loss / len(test_loader)
test_accuracy = test_correct / test_samples

print("Test Evaluation Completed!")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Correct Predictions: {test_correct}/{test_samples}")

Testing:   0%|          | 0/1563 [00:00<?, ?it/s]

Test Evaluation Completed!
Test Loss: 0.3399
Test Accuracy: 86.85%
Correct Predictions: 21712/25000


In [ ]:

import os

SAVE_DIR = "/content/SentimentScope_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save training configuration and test results
checkpoint = {
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
}

torch.save(
    checkpoint,
    os.path.join(SAVE_DIR, "sentimentscope_checkpoint.pt")
)

print("Model saved successfully!")
print("Saved location:", SAVE_DIR)
print("Test Accuracy:", f"{test_accuracy * 100:.2f}%")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Saved location: /content/SentimentScope_model
Test Accuracy: 86.85%


In [ ]:

import shutil

zip_path = shutil.make_archive(
    "/content/SentimentScope_submission",
    "zip",
    SAVE_DIR
)

print("Submission ZIP created:", zip_path)

Submission ZIP created: /content/SentimentScope_submission.zip


In [ ]:

import os
import shutil

SAVE_DIR = "/content/SentimentScope_model"

print("Model folder exists:", os.path.exists(SAVE_DIR))

if os.path.exists(SAVE_DIR):
    print("Files inside model folder:")
    print(os.listdir(SAVE_DIR))
else:
    print("Model folder not found. Run the model-saving cell first.")

Model folder exists: True
Files inside model folder:
['model.safetensors', 'tokenizer.json', 'config.json', 'sentimentscope_checkpoint.pt', 'tokenizer_config.json']


In [ ]:

import os
import shutil
from google.colab import files

SAVE_DIR = "/content/SentimentScope_model"

if os.path.exists(SAVE_DIR):
    zip_path = shutil.make_archive(
        "/content/SentimentScope_submission",
        "zip",
        SAVE_DIR
    )
    print("ZIP file created successfully!")
    files.download(zip_path)
else:
    print("Model folder missing. Run the model-saving code first.")

ZIP file created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

from transformers import pipeline

sentiment_model = pipeline(
    "text-classification",
    model="/content/SentimentScope_model",
    tokenizer="/content/SentimentScope_model",
    device=0 if torch.cuda.is_available() else -1
)

review = "This movie was amazing and I really enjoyed it!"
result = sentiment_model(review)

print("Movie Review:", review)
print("Prediction:", result[0]["label"])
print("Confidence:", round(result[0]["score"] * 100, 2), "%")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Movie Review: This movie was amazing and I really enjoyed it!
Prediction: LABEL_1
Confidence: 99.64 %


In [ ]:

import os

print("Current folders:")
print(os.listdir("/content"))

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".safetensors", ".pt", ".bin")):
            print(os.path.join(root, file))

Current folders:
['.config', 'drive', 'sample_data']


In [ ]:

import os

print(os.listdir("/content"))

['.config', 'drive', 'sample_data']
